# 6.2 Evals: golden datasets, LLM-as-judge

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

**Prerequisites:** 3.5 (golden sets and RAGAS metrics for RAG; this lesson generalises the same ideas to agents). The offline harness runs with no API key. The LangSmith `evaluate()` section and the real LLM-judge section need `LANGSMITH_API_KEY` and `OPENAI_API_KEY` respectively; everything else, including the judge mechanics, runs offline on a deterministic stub judge.

## Learning objectives

- Build a golden dataset for an agent: rows that capture expected *behaviour* (which tool, what action, when to abstain), not just expected text.
- Write pure-function evaluators (exact match, contains, trajectory checks) and run them offline before ever touching a hosted eval service.
- Run the same evaluators through LangSmith `evaluate()` with the current SDK signature, and read the experiment results.
- Design a criteria-based LLM-as-judge rubric, choose between absolute and pairwise scoring, and name the judge's biases with a mitigation for each.
- Turn the eval set into a regression gate: when to run it, what a red run means, and how to budget the judge's token cost.

## The demo-to-incident gap

Every agent demo looks magical and every production incident report starts with "it worked in the demo". The gap between the two is not effort or talent; it is measurement. In the demo you watched five queries. In production the agent meets the sixth query, the one with the typo, the angry customer, the edge case your prompt never considered, and it does something you never saw in testing.

Evals are the only thing standing between those two worlds. Not vibes, not a longer prompt, not a bigger model. A repeatable set of checks that tells you, on every change, whether the agent still does what it did yesterday. Lesson 3.5 built this habit for RAG pipelines. Agents need it more, because an agent does not just produce text: it calls tools, spends money, and takes actions. An eval set for an agent therefore checks behaviour, not just wording.

## Golden datasets for agents

A golden dataset is a list of examples with known-correct outcomes, curated by a human. For an agent, each row needs more than question and answer. The minimum useful schema:

In [ ]:
GOLDEN = [
    {
        "question": "What is the refund window?",
        "expected_behavior": "states the 30-day refund window",
        "must_contain": ["30", "day"],
        "must_not_call": ["issue_refund"],   # a question is not a refund request
    },
    {
        "question": "Refund order ORD-1042, it arrived broken.",
        "expected_behavior": "calls issue_refund with order_id ORD-1042",
        "expected_tool": "issue_refund",
        "must_not_call": [],
    },
    {
        "question": "How do I reset my password?",
        "expected_behavior": "abstains: says it does not have that information",
        "must_contain": ["do not have"],
        "must_not_call": ["issue_refund", "lookup_order"],
    },
]

Three design rules, each learned the hard way by teams that skipped them:

1. **Test behaviour, not prose.** `must_contain: ["30", "day"]` survives a reworded answer; an exact-string match does not. For tool-calling agents, the row should name the expected tool and its arguments, because calling the wrong tool is a worse failure than a clumsy sentence.
2. **Include the abstain rows.** Real users ask things the agent cannot do. A golden set with no "should refuse" rows trains your eval to reward confident guessing. Lesson 3.4 taught abstention for RAG; agents need the same rows, plus "should not call any tool" rows.
3. **Write rows from real traces, then anonymise.** The best questions come from production logs, beta transcripts, and support tickets (lesson 3.5). Strip PII before they enter the set: names, emails, order numbers become `ORD-1042` style placeholders. A golden set built from raw user queries is personal data under GDPR, and lesson 6.5 returns to the compliance side. Synthetic rows you write yourself carry none of this baggage, so start there if you have no logs yet.

**How many rows?** 20 to 50 catches regressions on a single agent during development: enough to cover the happy paths, the abstains, and a few adversarial cases, small enough to run in seconds offline. For a release gate, teams grow to a few hundred, sampled across the real query distribution. Past that, returns diminish: a 5,000-row set that nobody curates rots, because the product changed and the "golden" answers are now wrong. This is golden rot, and it is the most common way eval sets die. Review the set whenever the product changes, and delete rows that no longer describe correct behaviour.

## The offline harness first

Before LangSmith, before judges, write the harness as plain Python. Evaluators are pure functions of `(output, expected)`, so they run anywhere: your laptop, CI, a plane. Here is the complete runnable core, using the stub agent and five golden rows from this lesson's theme (a shop support agent):

In [ ]:
# pip install langchain==1.4.6  (no API key needed for this section)

def target_agent(question: str) -> str:
    """Stub for the agent under test. Swap in your real agent later;
    the harness does not change."""
    db = {
        "refund window": "You can request a refund within 30 days of purchase.",
        "vat rate": "The standard VAT rate is 25%.",
        "support hours": "Support is open Monday to Friday, 9:00 to 17:00 CET.",
    }
    q = question.lower()
    for key, answer in db.items():
        if key in q:
            return answer
    return "I do not have that information."

GOLDEN = [
    {"question": "What is the refund window?", "expected": "30 days", "must_contain": ["30", "day"]},
    {"question": "What VAT rate applies?", "expected": "25%", "must_contain": ["25"]},
    {"question": "What are the support hours?", "expected": "Mon-Fri 9-17 CET", "must_contain": ["monday", "friday"]},
    {"question": "How do I reset my password?", "expected": "ABSTAIN", "must_contain": ["do not have"]},
    {"question": "Do you refund after 60 days?", "expected": "no", "must_contain": ["30"]},
]

def exact_match(output: str, expected: str) -> dict:
    return {"key": "exact_match",
            "score": int(output.strip().lower() == expected.strip().lower())}

def contains_all(output: str, must_contain: list) -> dict:
    low = output.lower()
    missing = [t for t in must_contain if t.lower() not in low]
    return {"key": "contains", "score": int(not missing),
            "comment": f"missing={missing}" if missing else "all present"}

def run_eval() -> list[dict]:
    rows = []
    for ex in GOLDEN:
        out = target_agent(ex["question"])
        em = exact_match(out, ex["expected"])
        c = contains_all(out, ex["must_contain"])
        rows.append({"question": ex["question"], "output": out,
                     "exact_match": em["score"], "contains": c["score"]})
    return rows

if __name__ == "__main__":
    rows = run_eval()
    for r in rows:
        print(f"{r['question'][:38]:40} exact={r['exact_match']} contains={r['contains']}")
    passed = sum(r["contains"] for r in rows)
    print(f"\ncontains pass rate: {passed}/{len(rows)}")

Run it. Expected output:

In [ ]:
What is the refund window?               exact=0 contains=1
What VAT rate applies?                   exact=0 contains=1
What are the support hours?              exact=0 contains=1
How do I reset my password?              exact=0 contains=1
Do you refund after 60 days?             exact=0 contains=0
...
contains pass rate: 4/5

Read that output like an eval engineer. `exact_match` is 0 everywhere, which tells you exact match is the wrong evaluator for natural-language answers, not that the agent failed. The `contains` evaluator passes 4 of 5. The failing row is the interesting one: "Do you refund after 60 days?" is a paraphrase of the refund-window question, but the stub's crude keyword matching does not recognise it, so it abstains instead of answering "no, the window is 30 days". The eval caught a real capability gap: the agent handles the phrasing in the golden set but not the phrasing a user would actually type. That gap between the rows you wrote and the queries users write is exactly what a good golden set closes, by including paraphrases and adversarial phrasings alongside the clean questions.

A note on the abstain row: the stub returns "I do not have that information." and `contains` checks for "do not have". Testing abstention with a substring check is crude but honest, and it runs in a millisecond with no API bill.

## Trajectory evaluators: checking what the agent *did*

For tool-calling agents, the most valuable evaluator does not read the final text at all. It reads the trajectory: which tools were called, in what order, with what arguments. A support agent that answers "refund issued" without calling `issue_refund` is lying; one that calls it twice is a bug. The evaluator takes the list of tool calls from the run:

In [ ]:
def expected_tool_called(tool_calls: list[dict], expected_tool: str) -> dict:
    names = [c["name"] for c in tool_calls]
    return {"key": "expected_tool_called",
            "score": int(expected_tool in names),
            "comment": f"called={names}"}

def no_forbidden_tools(tool_calls: list[dict], forbidden: list[str]) -> dict:
    names = [c["name"] for c in tool_calls]
    bad = [n for n in names if n in forbidden]
    return {"key": "no_forbidden_tools",
            "score": int(not bad),
            "comment": f"forbidden_called={bad}" if bad else "clean"}

Where do `tool_calls` come from? From your agent's trace: with LangGraph, the messages in the final state carry `tool_calls` on each `AIMessage`; with `create_agent`, the returned state's messages give you the same. Lesson 6.1 covers pulling trajectories from LangSmith traces; the evaluator shape above works with either source. Trajectory checks are deterministic, cheap, and immune to judge bias, so use them wherever the requirement is behavioural ("called the right tool") and reserve the LLM judge for the genuinely linguistic requirements ("the tone was appropriate").

## Running it in LangSmith

The offline harness is the daily driver. LangSmith's `evaluate()` is where the same checks become a tracked experiment you can compare across prompt versions and models. The current signature on the pinned SDK (verified on langsmith 0.14.12):

In [ ]:
from langsmith.evaluation import evaluate

experiment_results = evaluate(
    target_fn,                       # your agent: inputs dict -> outputs dict
    data="support-golden-v1",        # dataset name in LangSmith
    evaluators=[exact_match_ls, contains_ls],
    experiment_prefix="support-agent",
    metadata={"prompt_version": "v3", "model": "gpt-4o-mini"},
    max_concurrency=4,
    num_repetitions=1,
)

`target_fn` receives each example's inputs and returns an outputs dict; each evaluator receives `(run, example)` and returns a `{"key", "score", "comment"}` dict, the same shape as the offline evaluators above. That shape compatibility is deliberate: write the scoring logic once as a pure function, then wrap it thinly for LangSmith:

In [ ]:
from langsmith.schemas import Run, Example

def contains_ls(run: Run, example: Example) -> dict:
    output = run.outputs["answer"]
    must = example.outputs["must_contain"]
    result = contains_all(output, must)   # the same pure function as offline
    return {"key": result["key"], "score": result["score"], "comment": result["comment"]}

Each run creates a named experiment; re-running after a prompt change with the same dataset and a new `metadata` tag gives you a side-by-side comparison in the LangSmith UI. `num_repetitions` re-runs each example N times, which is how you separate a real regression from one unlucky sample.

This section needs `LANGSMITH_API_KEY`. If you are following along offline, the harness from the previous section is the same logic with no key; nothing about the evaluators changes when you move them to LangSmith.

## LLM-as-judge: the biased grader you still need

Some requirements resist pure functions. "The answer is polite", "the summary captures the nuance", "the refusal explains the policy". For those, the industry-standard tool is LLM-as-judge: a second model scores the first model's output against a rubric. It works, and it is biased in documented ways. Both facts matter.

**Absolute scoring** is the default: the judge sees the question, a reference answer, the candidate answer, and a rubric, and returns a score. Keep the rubric narrow and the output structured:

In [ ]:
RUBRIC = """Score the answer 0-2 against the reference.
2 = fully correct and directly answers the question.
1 = partially correct or correct but evasive.
0 = wrong, or answers a different question.
Consider ONLY correctness. Length is not quality.
Return ONLY the integer."""

The offline version in this lesson uses a deterministic stub judge (keyword overlap against the reference) so the whole pipeline runs without a key. The production version swaps the stub for a real model with structured output:

In [ ]:
# Needs OPENAI_API_KEY. Same rubric, real judge.
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI   # langchain-openai==1.7.2

class Verdict(BaseModel):
    score: Literal[0, 1, 2]
    reason: str = Field(max_length=300)

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Verdict)

def llm_judge(question: str, output: str, expected: str) -> dict:
    verdict = judge.invoke(
        f"{RUBRIC}\n\nQuestion: {question}\nReference: {expected}\nAnswer: {output}"
    )
    return {"key": "judge_correctness", "score": verdict.score / 2,
            "comment": verdict.reason}

`temperature=0` is not optional for a grader: you want determinism, not creativity. Pin the judge model exactly, because a judge whose model drifts under you is a ruler that changes length.

**Pairwise scoring** is the alternative: the judge sees two answers and picks the better one. It is useful for comparing two agent versions head to head, but it carries position bias, so the honest protocol runs each pair twice with the order swapped and averages. Absolute scoring with a good rubric is the better default for regression gates; pairwise is the better tool for "which of these two prompts is better".

**The biases, with mitigations.** These are measured effects from the literature, not folklore:

| Bias | What happens | Mitigation |
|---|---|---|
| Verbosity | longer answers score higher even when the extra words add nothing | rubric states length is not quality; score only the listed criteria |
| Position | in pairwise tests the first (or last) answer wins regardless of quality | run both orders and average; prefer absolute scoring |
| Self-preference | a judge favours outputs written in its own model family's style; one study measured a pairwise win rate above 0.90 for a model judging its own summaries | use a judge from a different family than the agent under test |
| Leniency | unsure judges default to PASS | calibrate: the judge must earn trust against your labels (below) |
| Non-determinism | the same input can score differently across runs | temperature 0, pinned model, `num_repetitions` to measure the spread |

**Calibration is the whole game.** Before trusting a judge, score 30 to 50 examples yourself, run the judge on the same set, and compute agreement (Cohen's kappa for categorical verdicts, or simple accuracy on pass/fail). If the judge disagrees with you more than about 10 to 15% of the time, fix the rubric, not the agent. A judge you never calibrated is a random number generator with good marketing. Practitioners on social channels converge on the same five-step checklist: ground the judge with question, reference and rubric; use structured outputs; validate against human labels; test for bias; keep a human in the loop for high-stakes cases.

## Regression testing in CI

An eval set that only runs when someone remembers to run it is documentation, not a gate. Wire it into CI at three points:

1. **On prompt or agent-code change:** run the offline harness (seconds, free). Fail the build below a threshold, for example `contains pass rate >= 0.8` and `judge average >= 0.75`. The threshold lives in the repo next to the golden set so a change to either is reviewed together.
2. **On model upgrade:** run the full set with the real judge, `num_repetitions=3`, and compare against the previous model's experiment in LangSmith. Model upgrades are the silent killer: same prompt, different behaviour, nobody ran the evals.
3. **Nightly:** run the full set against production traces sampled that day, scored by the cheap evaluators. This catches drift in the world (new product names, changed policies) rather than drift in your code.

**Red versus flaky.** A red eval run means one of three things: the agent regressed, the golden set rotted, or the judge flaked. Tell them apart before reacting. Re-run with a fixed seed and `num_repetitions=3`: if the failure is stable across repetitions, it is real; if it flickers, it is judge noise, and the fix is a tighter rubric or a cheaper deterministic evaluator for that row, not a prompt change. Never "fix" a flaky judge by editing the agent until the noise stops; you will tune the agent to the noise.

In [ ]:
# tests/test_agent_regression.py  (runs in CI, no API key)
from eval_harness import run_eval

def test_golden_set_regression():
    rows = run_eval()
    contains_rate = sum(r["contains"] for r in rows) / len(rows)
    assert contains_rate >= 0.8, f"regression: contains pass rate {contains_rate:.2f}"
    # exact_match is recorded for information, not gated: it is 0 on
    # reworded-but-correct answers by design (see the lesson section above).

## The cost of evals

Judge calls add up faster than agent calls, because each eval row costs at least one judge call and often several (repetitions, pairwise orders, multiple criteria). Budget them like any other line item:

| Setup | Judge calls per run | Rough cost driver |
|---|---|---|
| 50 rows, deterministic evaluators only | 0 | free; run on every commit |
| 50 rows, 1 judge, absolute scoring | 50 | one model call per row |
| 200 rows, 1 judge, 3 repetitions | 600 | repetitions multiply everything |
| 200 rows, pairwise A/B, both orders | 400 | pairwise doubles the bill |

Two rules keep the bill sane. First, gate cheap: deterministic evaluators run on every change for free, and the judge only runs when the cheap gate passes or on a schedule. Second, sample the expensive checks: run the full judge suite nightly, not per commit. A 200-row nightly judge run at a few cents per row is a rounding error next to one production incident.

## EU angle: evals as compliance evidence

If you deploy an agent in the EU, evals stop being good practice and start being evidence. The EU AI Act places accuracy and robustness obligations on deployers of AI systems: you are expected to be able to show that the system behaves as intended, including under reasonably foreseeable misuse. A versioned golden set, tracked experiments in LangSmith with model and prompt metadata, and recorded judge calibrations are exactly the artefacts an auditor asks for: what you tested, what changed, and what the numbers were. Keep the experiment metadata honest (model version, prompt version, date); an eval history with no versions proves nothing. Lesson 6.5 covers the full deployer picture, including data residency for the traces these experiments generate.

## What can go wrong

- **Golden rot.** The product changed, the golden answers did not, and now the eval fails on rows that describe the old world. Review the set on every product change; a failing row is guilty until proven innocent, but the row can be the guilty party.
- **Testing your own exhaust.** If the reference answers were generated by the agent under test, every metric congratulates you on being consistently wrong. References are written or checked by a human, never copied from pipeline output.
- **One evaluator, blind trust.** A single cheap check under-specifies most requirements: "contains 30" is not the same as "correctly answers the refund question". Layer evaluators: deterministic checks for behaviour, the judge for language, trajectory checks for tool use, and paraphrase rows in the golden set so keyword-matching agents cannot hide behind clean phrasings.
- **An uncalibrated judge.** A judge you never measured against your own labels will approve your worst bug with total confidence. Calibrate on 30 to 50 hand-labelled examples before the judge gates anything.
- **Judge bias becoming your bias.** Verbosity, position, self-preference, and leniency are measured effects. If your eval drives a release decision, the judge's biases are your release criteria. Debias with rubrics, swapped pairwise orders, cross-family judges, and spot checks.
- **Flaky red builds.** A judge that flickers makes CI red at random, and the team learns to ignore red builds, which defeats the gate. Separate stable failures (real) from flicker (judge noise) with repetitions before changing the agent.
- **Eval cost surprise.** A 200-row pairwise suite with repetitions can cost more than the agent's weekly inference. Gate cheap evaluators per commit; run the judge nightly.
- **PII in the golden set.** Rows built from real traces contain real personal data. Anonymise before the set enters the repo, keep raw traces out of version control, and check your data processing agreements if a hosted eval service sees them.

## Check your understanding

**1. Why should an agent's golden dataset test behaviour (which tool was called, when to abstain) rather than just the wording of the final answer?**
Because an agent's failures are behavioural: calling the wrong tool, calling a tool twice, or answering confidently when it should abstain. Two answers with identical wording can hide different trajectories, and the trajectory is where the money and the side effects live. Wording checks also break on harmless rephrasing, while behaviour checks survive it.

**2. The "refund after 60 days" row fails even though it is a paraphrase of a covered question. What does this failure tell you about golden set design?**
It tells you the set is testing the phrasings you wrote, not the phrasings users write. The stub handles "What is the refund window?" but not "Do you refund after 60 days?", and the eval caught the gap. The fix is in the dataset, not the evaluator: add paraphrases, typos, and adversarial rewordings of every key behaviour, because real user queries rarely match your clean questions word for word.

**3. Name three documented LLM-as-judge biases and one mitigation for each.**
Verbosity bias (longer answers score higher): state in the rubric that length is not quality and score only listed criteria. Position bias (first/last answer wins in pairwise tests): run both orders and average, or prefer absolute scoring. Self-preference (judge favours its own model family's style): use a judge from a different family than the agent under test. Leniency also accepted: calibrate the judge against human labels.

**4. A CI eval run goes red the morning after a model upgrade, with no prompt or code change. Walk through how you decide whether it is a real regression.**
First re-run with `num_repetitions=3` and a fixed seed: flickering failures are judge noise, stable ones are real. Then check golden rot: did the world change (new policy, new product name) so the reference is now wrong? Then compare the LangSmith experiments for the old and new model side by side on the same dataset. Only after those three checks do you touch the prompt or the agent.

**5. Why do deterministic evaluators run on every commit while the LLM judge runs nightly, and what does the EU AI Act have to do with any of this?**
Deterministic evaluators are free and stable, so they make a good per-commit gate; the judge costs a model call per row and can flake, so it runs on a schedule where its cost and noise are tolerable. The AI Act angle: deployers must be able to demonstrate accuracy and robustness, and a versioned golden set plus tracked experiments with model and prompt metadata is the evidence an auditor asks for.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.